## 8.4 学习率衰减策略 - ReduceLROnPlateau（最常用方式）

#### 1. 为什么需要基于性能的学习率衰减
在前面我们学习了三种学习率衰减方式：
```
方法	下降方式
StepLR	固定 epoch 下降
MultiStepLR	指定 epoch 下降
ExponentialLR	每个 epoch 指数下降
```
这些方法都有一个共同特点：

`学习率下降的时间点是提前设定好的`

也就是说：
* 不管模型学得怎么样
* 到时间就降学习率

但在真实训练中，经常会出现这样的情况：
* 模型 loss 已经不下降了
* 但是学习率还没有下降

或者：
* 模型仍然在快速学习
* 但学习率已经被降低

因此更合理的策略是：

`根据模型表现自动调整学习率`

也就是说：
* 当模型性能停止提升时
* 降低学习率

这就是 ReduceLROnPlateau 的思想。

#### 2. 什么是 ReduceLROnPlateau
ReduceLROnPlateau 的核心思想是：

`当模型性能在一段时间内没有提升时，自动降低学习率。`

例如：

监控指标：

`validation loss`

如果：
* 连续 5 个 epoch
* loss 没有下降

则：

`学习率 × 0.1`

例如：
```
Epoch	Val Loss	学习率
1	0.45	0.01
2	0.40	0.01
3	0.39	0.01
4	0.39	0.01
5	0.39	0.01
6	0.39	0.01
7	0.39	0.001
```
因为：
* loss 停止下降

所以：
* 学习率自动降低

#### 3. ReduceLROnPlateau 的核心机制
该方法包含三个核心概念：

##### 3.1 监控指标（monitor）
可以监控：
```
validation loss
validation accuracy
training loss
```

通常使用：

`validation loss`

##### 3.2 patience
表示：

`允许多少个 epoch 没有改善（Loss下降，Accuracy没有提升）`

例如：

`patience = 5`

表示：
* 5 个 epoch 内没有 improvement
* 才会降低学习率。

##### 3.3 factor
表示：

`学习率下降比例`

例如：

`factor = 0.1`

表示：

`lr = lr × 0.1`

#### 4. 学习率变化示例
假设：
```
初始 lr = 0.01
factor = 0.1
patience = 3
```

验证 loss：
```
Epoch	Val Loss	学习率
1	0.50	0.01
2	0.42	0.01
3	0.41	0.01
4	0.41	0.01
5	0.41	0.01
6	0.41	0.001
```
因为：
* 3 个 epoch 没有 improvement

触发：
* 学习率下降

#### 5. ReduceLROnPlateau 的优点
**1️⃣ 自动化程度高**

不需要人工设定：

第几个 epoch 降学习率

算法会自动判断。

**2️⃣ 更符合训练实际情况**

学习率下降的时机取决于：
* 模型性能

而不是：
* 第几个Epoch

**3️⃣ 实践中非常常用**

在很多实际深度学习训练中：

`ReduceLROnPlateau`

是最常用的学习率策略之一

特别是在：
```
CNN
Transformer
医学影像
```

#### 6. PyTorch 中的实现
在 PyTorch 中使用：

`torch.optim.lr_scheduler.ReduceLROnPlateau`

#### 6.1 创建优化器

In [ ]:
import torch 
model = torch.nn.Linear(10, 5)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

##### 6.2 创建学习率调度器

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,
    patience=10,
    verbose=True
)

##### 6.3 训练循环
⚠️注意：ReduceLROnPlateau 和其他 scheduler 不一样

它需要传入：
* 验证指标
* 根据mode 参数来选择，通常是：validation_loss

因此在训练过程中，我们需要 计算验证损失。

1. 正常训练过程

训练方法作为正常的前向+ 反向传播方法

不作为ReduceLROnPlateau 的Measure

In [ ]:
def train(model, dataloader, optimizer, criterion):

    model.train()

    total_loss = 0

    for x, y in dataloader:

        optimizer.zero_grad()

        y_pred = model(x)

        loss = criterion(y_pred, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(dataloader)

    return train_loss

2. 计算验证损失

验证阶段使用 验证数据集（validation dataset），

并且不进行反向传播，只是作为ReduceLROnPlateau的Measure

In [ ]:
def validate(model, dataloader, criterion):

    model.eval()

    total_loss = 0

    with torch.no_grad():

        for x, y in dataloader:

            y_pred = model(x)

            loss = criterion(y_pred, y)

            total_loss += loss.item()

    val_loss = total_loss / len(dataloader)

    return val_loss

3. 训练主循环

In [ ]:
for epoch in range(epochs):

    train_loss = train(model, train_loader, optimizer, criterion)

    val_loss = validate(model, val_loader, criterion)

    scheduler.step(val_loss)